# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ni7chal14/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

Lane: “Which content items should be reviewed first?” Week 4 created a transparent ranking rule. We compare it with two learned ranking scores: Logistic Regression (a readable first classifier) and a shallow Decision Tree (a small, inspectable nonlinear model).

The target is the observed is_declining_label (1 when trend_direction == 'down'). This is a retrospective ranking exercise, not proof of future forecasting ability. We evaluate precision@20 and precision@50 on held-out clients and show the test-set base rate.

To avoid target leakage, trend_direction and trend_pct are excluded. We also exclude last-30-day and previous-30-day metrics because comparing those periods could reconstruct the trend used to create the label. content_id and client_id are identifiers, not predictive features. avg_position == 0 is treated as missing, per the data dictionary.

We fix the random seed and use a client-grouped split. The Week 4 rule is frozen: stale × visible × impressions_90d, where stale means days_since_last_update >= 91 and visible means impressions_90d >= 300.
Sensitivity check added: because the starter file is a single trailing-90-day snapshot and the label is based on trend_pct, the main model is a retrospective association model. We also fit a conservative model using content/keyword metadata and age/freshness context only, excluding current performance metrics and derived performance tiers. This is a robustness check, not proof of future predictive ability.

In [1]:
# Load data and define the observed target.
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn

SEED = 42
DATA_PATHS = [
    Path('data/raw/content_refresh_anonymized.csv'),
    Path('/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv'),
]
data_path = next((p for p in DATA_PATHS if p.exists()), None)
if data_path is not None:
    df = pd.read_csv(data_path)
else:
    # Used when this notebook is opened directly from the public GitHub repo in Colab.
    raw_url = 'https://raw.githubusercontent.com/Ni7chal14/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv'
    df = pd.read_csv(raw_url)

required = {'client_id', 'trend_direction', 'days_since_last_update', 'impressions_90d'}
missing_required = required - set(df.columns)
if missing_required:
    raise ValueError(f'Dataset is missing required columns: {sorted(missing_required)}')

df['is_declining_label'] = (df['trend_direction'] == 'down').astype('int8')
print(f'Rows: {len(df):,} | clients: {df.client_id.nunique()} | columns: {df.shape[1]}')
print(f'Target base rate (all rows, descriptive only): {df.is_declining_label.mean():.3f}')
print(f'pandas={pd.__version__} | scikit-learn={sklearn.__version__}')
df.head(3)

Rows: 30,000 | clients: 32 | columns: 45
Target base rate (all rows, descriptive only): 0.542
pandas=2.2.3 | scikit-learn=1.6.1


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1


## 2. Split design

We use GroupShuffleSplit with client_id as the group and a fixed seed. This tests generalization to content from clients the model did not train on, rather than letting pages from one client appear on both sides.

There are only 32 clients, so the test set contains a subset of clients and metrics may vary with the split. We report the actual train/test client counts and the test-set base rate. All methods—including the Week 4 rule—are evaluated on this exact same test set.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

# IDs and label-derived / label-reconstructing columns must not be features.
EXCLUDED_COLUMNS = {
    'content_id', 'client_id', 'trend_direction', 'trend_pct',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
    'is_declining_label',
}
feature_cols = [c for c in df.columns if c not in EXCLUDED_COLUMNS]
model_df = df.copy()
model_df['avg_position'] = model_df['avg_position'].replace(0, np.nan)
X = model_df[feature_cols].copy()
y = model_df['is_declining_label'].astype(int)
groups = model_df['client_id']

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
test_rows = df.iloc[test_idx].copy()
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
assert train_clients.isdisjoint(test_clients), 'Client leakage: client appears in both splits.'

print(f'Train rows: {len(train_idx):,} | Test rows: {len(test_idx):,}')
print(f'Train clients: {len(train_clients)} | Test clients: {len(test_clients)}')
print(f'Test-set decline base rate: {y_test.mean():.3f}')
print(f'Features used: {len(feature_cols)}')
print('Excluded columns:', sorted(EXCLUDED_COLUMNS))

Train rows: 23,837 | Test rows: 6,163
Train clients: 25 | Test clients: 7
Test-set decline base rate: 0.511
Features used: 34
Excluded columns: ['clicks_last_30d', 'clicks_prev_30d', 'client_id', 'content_id', 'impressions_last_30d', 'impressions_prev_30d', 'is_declining_label', 'sessions_last_30d', 'sessions_prev_30d', 'trend_direction', 'trend_pct']


## 3. Train + compare vs my baseline

We fit Logistic Regression and a depth-2 Decision Tree. Numeric missing values are median-imputed and categorical missing values are imputed with the most frequent category; categoricals are one-hot encoded. Preprocessing is fitted on the training split only.

The baseline and both models are ranked on the same held-out rows. The main question is how many items in the top 20 or top 50 are labeled declining, so we use precision@K rather than ordinary accuracy. This is retrospective evaluation against the observed label—not a validated future forecast.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

numeric_cols = X_train.select_dtypes(include=['number', 'bool']).columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]
numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler()),
])
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipe, numeric_cols),
    ('cat', categorical_pipe, categorical_cols),
])
models = {
    'Logistic Regression': LogisticRegression(max_iter=1500, class_weight='balanced', random_state=SEED),
    'Decision Tree (depth=2)': DecisionTreeClassifier(max_depth=2, min_samples_leaf=100, class_weight='balanced', random_state=SEED),
}
fitted_models, scores = {}, {}
for name, estimator in models.items():
    pipe = Pipeline([('preprocess', preprocessor), ('model', estimator)])
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe
    scores[name] = pipe.predict_proba(X_test)[:, 1]

# Frozen Week-4 rule, recomputed on the same held-out rows.
stale = (test_rows['days_since_last_update'].fillna(-1) >= 91).astype(int)
visible = (test_rows['impressions_90d'].fillna(0) >= 300).astype(int)
baseline_score = (stale * visible * test_rows['impressions_90d'].fillna(0)).to_numpy()

def precision_at_k(y_true, ranking_scores, k):
    y_array = np.asarray(y_true, dtype=int)
    score_array = np.asarray(ranking_scores, dtype=float)
    k_actual = min(k, len(y_array))
    order = np.argsort(-score_array, kind='stable')[:k_actual]
    return float(y_array[order].mean()), k_actual

rows = []
for name, ranking_score in [('Week-4 rule baseline', baseline_score)] + list(scores.items()):
    p20, n20 = precision_at_k(y_test, ranking_score, 20)
    p50, n50 = precision_at_k(y_test, ranking_score, 50)
    rows.append({'method': name, 'precision@20': p20, 'precision@50': p50,
                 'base_rate': float(y_test.mean()), 'items@20': n20, 'items@50': n50})
comparison = pd.DataFrame(rows).sort_values('precision@50', ascending=False)
display(comparison.round(3))
print('Precision@K is the fraction of the top K ranked items labeled declining.')
print('The base rate is the expected precision of random selection on this same test set.')

,method,precision@20,precision@50,base_rate,items@20,items@50
1,Logistic Regression,0.6,0.68,0.511,20,50
2,Decision Tree (depth=2),0.7,0.64,0.511,20,50
0,Week-4 rule baseline,0.2,0.30,0.511,20,50


Precision@K is the fraction of the top K ranked items labeled declining.
The base rate is the expected precision of random selection on this same test set.


##3B. Leakage-sensitivity check: conservative features only
The full model can use contemporaneous performance measures that may be related to the way the observed decline label was constructed. To see how dependent the results are on those measures, this second experiment uses a deliberately restricted allowlist of content/keyword metadata plus age/freshness context. It excludes traffic, search-performance, engagement, AI-traffic, and derived performance-tier fields.

Both conservative models use the same client-grouped train/test split and are evaluated with the same precision@20 and precision@50 metrics. This comparison diagnoses sensitivity to feature choice; it does not establish a true future forecast. The Week-4 baseline is included again on the same test rows.

In [4]:
# Conservative feature allowlist: metadata/context only; no current performance metrics.
CONSERVATIVE_ALLOWLIST = [
    'content_type', 'main_intent',
    'search_volume', 'competition', 'competition_level', 'cpc',
    'word_count', 'char_count',
    'content_age_days', 'days_since_last_update',
]
conservative_features = [c for c in CONSERVATIVE_ALLOWLIST if c in df.columns]
missing_conservative = sorted(set(CONSERVATIVE_ALLOWLIST) - set(df.columns))
if not conservative_features:
    raise ValueError('None of the conservative allowlist columns exist in this dataset.')

# Use the exact same row indices as the primary experiment.
X_cons = df[conservative_features].copy()
if 'avg_position' in X_cons.columns:
    X_cons['avg_position'] = X_cons['avg_position'].replace(0, np.nan)
X_cons_train = X_cons.iloc[train_idx].copy()
X_cons_test = X_cons.iloc[test_idx].copy()

cons_numeric = X_cons_train.select_dtypes(include=['number', 'bool']).columns.tolist()
cons_categorical = [c for c in X_cons_train.columns if c not in cons_numeric]
cons_numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler()),
])
cons_categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])
cons_preprocessor = ColumnTransformer([
    ('num', cons_numeric_pipe, cons_numeric),
    ('cat', cons_categorical_pipe, cons_categorical),
])
conservative_models = {
    'Conservative Logistic Regression': LogisticRegression(
        max_iter=1500, class_weight='balanced', random_state=SEED
    ),
    'Conservative Decision Tree (depth=2)': DecisionTreeClassifier(
        max_depth=2, min_samples_leaf=100, class_weight='balanced', random_state=SEED
    ),
}
conservative_scores = {}
for name, estimator in conservative_models.items():
    pipe = Pipeline([('preprocess', cons_preprocessor), ('model', estimator)])
    pipe.fit(X_cons_train, y_train)
    conservative_scores[name] = pipe.predict_proba(X_cons_test)[:, 1]

sensitivity_rows = []
all_rankings = [('Week-4 rule baseline', baseline_score)] + list(scores.items()) + list(conservative_scores.items())
for name, ranking_score in all_rankings:
    p20, n20 = precision_at_k(y_test, ranking_score, 20)
    p50, n50 = precision_at_k(y_test, ranking_score, 50)
    sensitivity_rows.append({
        'method': name,
        'precision@20': p20,
        'precision@50': p50,
        'test_base_rate': float(y_test.mean()),
        'items@20': n20,
        'items@50': n50,
    })
sensitivity_comparison = pd.DataFrame(sensitivity_rows)
display(sensitivity_comparison.sort_values('precision@50', ascending=False).round(3))
print('Conservative features used:', conservative_features)
if missing_conservative:
    print('Allowlist columns absent from this dataset (skipped):', missing_conservative)
print('Interpretation: compare full vs conservative performance. A drop suggests the full model depends on current performance signals; it does not by itself prove leakage.')

,method,precision@20,precision@50,test_base_rate,items@20,items@50
1,Logistic Regression,0.6,0.68,0.511,20,50
4,Conservative Decision Tree (depth=2),0.7,0.64,0.511,20,50
2,Decision Tree (depth=2),0.7,0.64,0.511,20,50
3,Conservative Logistic Regression,0.5,0.44,0.511,20,50
0,Week-4 rule baseline,0.2,0.30,0.511,20,50


Conservative features used: ['content_type', 'main_intent', 'search_volume', 'competition', 'competition_level', 'cpc', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update']
Interpretation: compare full vs conservative performance. A drop suggests the full model depends on current performance signals; it does not by itself prove leakage.


## 4. Errors and interpretation

We inspect Logistic Regression errors rather than trusting the score alone. A false positive is classified as declining at the 0.5 threshold but has label 0; a false negative has label 1 but is classified as not declining. These are difficult cases for a retrospective model and may also reflect limitations of the label.

Permutation importance on a sample of held-out rows is a diagnostic, not proof of causation. If a feature looks suspiciously perfect, re-check whether it leaks the target or its construction.

In [5]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score

primary_model = fitted_models['Logistic Regression']
primary_scores = scores['Logistic Regression']
predicted = (primary_scores >= 0.5).astype(int)
y_test_array = y_test.to_numpy()
error_view = pd.DataFrame({
    'observed_decline': y_test_array,
    'predicted_decline_at_0.5': predicted,
    'predicted_probability': primary_scores,
    'error_type': np.where((predicted == 1) & (y_test_array == 0), 'false_positive',
                    np.where((predicted == 0) & (y_test_array == 1), 'false_negative', 'correct')),
    'days_since_last_update': test_rows['days_since_last_update'].to_numpy(),
    'impressions_90d': test_rows['impressions_90d'].to_numpy(),
    'content_type': test_rows['content_type'].astype(str).to_numpy(),
})
print('Error counts at probability threshold 0.5:')
display(error_view['error_type'].value_counts().rename_axis('error_type').to_frame('count'))
errors = error_view[error_view['error_type'] != 'correct'].copy()
if len(errors):
    examples = pd.concat([
        errors[errors.error_type == 'false_positive'].sort_values('predicted_probability', ascending=False).head(2),
        errors[errors.error_type == 'false_negative'].sort_values('predicted_probability', ascending=True).head(2),
    ]).head(3)
    display(examples.round(3))
else:
    print('No threshold-based errors on this split; inspect top-K ranked cases as a next check.')

# Permutation importance on up to 1,500 held-out examples for manageable runtime.
sample_n = min(1500, len(X_test))
X_imp = X_test.sample(n=sample_n, random_state=SEED)
y_imp = y_test.loc[X_imp.index]
perm = permutation_importance(primary_model, X_imp, y_imp, scoring='average_precision',
                              n_repeats=3, random_state=SEED, n_jobs=-1)
importance = pd.DataFrame({'feature': X_imp.columns,
                           'importance_mean': perm.importances_mean,
                           'importance_std': perm.importances_std}).sort_values('importance_mean', ascending=False)
display(importance.head(10).round(4))
print(f'Logistic Regression average precision on held-out clients: {average_precision_score(y_test, primary_scores):.3f}')
print('Importance indicates association, not causation. Treat results as directional: only 32 client groups are available.')

Error counts at probability threshold 0.5:


,count
error_type,
correct,3454
false_positive,1617
false_negative,1092


,observed_decline,predicted_decline_at_0.5,predicted_probability,error_type,days_since_last_update,impressions_90d,content_type
4309,0,1,0.888,false_positive,104,3115,keyword article
3836,0,1,0.878,false_positive,104,166,keyword article
5613,1,0,0.017,false_negative,92,1,keyword article


,feature,importance_mean,importance_std
28,avg_position,0.0475,0.0042
18,days_with_impressions,0.0355,0.0051
19,days_with_sessions,0.0348,0.0048
20,content_age_days,0.0283,0.0033
14,users_90d,0.0249,0.0029
6,word_count,0.0148,0.0049
22,age_tier_order,0.0086,0.0021
29,engagement_rate,0.0064,0.0018
32,impression_tier,0.0046,0.0008
11,clicks_90d,0.0026,0.0014


Logistic Regression average precision on held-out clients: 0.575
Importance indicates association, not causation. Treat results as directional: only 32 client groups are available.


### Conclusion

The Decision Tree achieved the highest Precision@20 at 70%, while Logistic Regression achieved the highest Precision@50 at 68%. Both outperformed the Week 4 rule baseline on the held-out client test set, whose decline-label base rate was 51.1%.

The conservative-feature sensitivity check showed that Logistic Regression's Precision@50 decreased from 68% to 44%, suggesting that performance-related signals contribute to its ranking performance. The Decision Tree's reported precision remained unchanged. These results do not establish leakage, but the timing of the target label and feature windows should be verified before claiming future predictive capability.

The Logistic Regression error analysis identified both false positives and false negatives at the 0.5 probability threshold. Permutation importance highlighted average position, days with impressions, days with sessions, and content age as useful predictive signals. These associations are not causal explanations.

The results should be interpreted as directional because the evaluation uses only 32 client groups, with seven held out for testing. Further validation on additional clients and a clearly separated future target window would strengthen confidence in generalization.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.